In [6]:
import regex as re

In [7]:
class BasicTokenizer:
    def train(self, text, vocab_size, verbose=False):
        self.merges = dict()
        self.vocab = [bytes([i]) for i in range(256)]
        merge_num = vocab_size - 256
        byte_txt = list(text.encode("utf-8"))
        for i in range(merge_num):
            top_pair = count_pair(byte_txt)[0][0]
            self.vocab.append(self.vocab[top_pair[0]] + self.vocab[top_pair[1]])
            self.merges[top_pair] = i + 256
            byte_txt = self.merge(byte_txt, top_pair)
        
        self.rev_merges = {v: k for k, v in self.merges.items()}    

    def merge(self, text, pair):
        new_txt = []
        skip_next = False
        for i in range(len(text) - 1):
            if skip_next:
                skip_next = False
                continue
            if text[i:i+2] == list(pair):
                new_txt.append(self.merges[pair])
                skip_next = True
            else:
                new_txt.append(text[i])
        if not skip_next:
            new_txt.append(text[-1])
        return new_txt

    def encode(self, text):
        text = list(text.encode("utf-8"))
        for mer in self.merges:
            new_txt = []
            skip_next = False
            for pair in zip(text, text[1:]):
                if skip_next:
                    skip_next = False
                    continue
                if pair == mer:
                    new_txt.append(self.merges[pair])
                    skip_next = True
                else:
                    new_txt.append(pair[0])
            if not skip_next:
                new_txt.append(text[-1])
            text = new_txt.copy()

        return text

    def decode(self, ids):
        change = True
        while change:
            new_ids = []
            change = False
            for id in ids:
                if self.rev_merges.get(id):
                    new_ids += list(self.rev_merges[id])
                    change = True
                else:
                    new_ids.append(id)

            
            ids = new_ids.copy()
        text = bytes(ids).decode("utf-8")

        return text

    def visualizer_merges(self):
        print(bytes(self.merges[0][0]) + " + " + bytes(self.merges[0][1]))

In [9]:
def count_pair(text):
    """text in bytes"""
    count = dict()
    for a, b in zip(text, text[1:]):
        count[(a, b)] = count.get((a,b), 0) + 1

    return sorted(count.items(), key=lambda x: x[1], reverse=True)

In [34]:
tokenizer = BasicTokenizer()
tokenizer.train("hello uw ơ ????", 260)
tokenizer.merges

{(63, 63): 256, (104, 101): 257, (257, 108): 258, (258, 108): 259}

In [11]:
def count_pair_lst(byte_txt_lst):
    count = dict()
    for text in byte_txt_lst:
        for a, b in zip(text, text[1:]):
            count[(a, b)] = count.get((a,b), 0) + 1
    return sorted(count.items(), key=lambda x: x[1], reverse=True)

In [12]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

pattern = re.compile(GPT4_SPLIT_PATTERN)
text = "aa aa bb ??"
splitted = re.findall(pattern, text)
byte_txt = [t.encode("utf-8") for t in re.findall(pattern, text)]
count_pair_lst(byte_txt)

[((97, 97), 2),
 ((32, 97), 1),
 ((32, 98), 1),
 ((98, 98), 1),
 ((32, 63), 1),
 ((63, 63), 1)]

In [88]:
class RegexTokenizer:
    def train(self, text, vocab_size, verbose=False):
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        pattern = re.compile(GPT4_SPLIT_PATTERN)
        
        self.merges = dict()
        self.vocab = [bytes([i]) for i in range(256)]
        merge_num = vocab_size - 256
        byte_txt = [t.encode("utf-8") for t in re.findall(pattern, text)]
        for i in range(merge_num):
            top_pair = self.count_pair_lst(byte_txt)[0][0]
            self.vocab.append(self.vocab[top_pair[0]] + self.vocab[top_pair[1]])
            self.merges[top_pair] = i + 256
            byte_txt = self.merge_lst(byte_txt, top_pair)
        
        self.rev_merges = {v: k for k, v in self.merges.items()}

    def count_pair_lst(self, byte_txt_lst):
        count = dict()
        for text in byte_txt_lst:
            for a, b in zip(text, text[1:]):
                count[(a, b)] = count.get((a,b), 0) + 1
        return sorted(count.items(), key=lambda x: x[1], reverse=True)

    def merge_lst(self, byte_txt_lst, pair):
        new_lst = []
        for text in byte_txt_lst:
            new_txt = []
            text = list(text)
            skip_next = False
            for i in range(len(text) - 1):
                if skip_next:
                    skip_next = False
                    continue
                if text[i:i+2] == list(pair):
                    new_txt.append(self.merges[pair])
                    skip_next = True
                else:
                    new_txt.append(text[i])
            if not skip_next:
                new_txt.append(text[-1])
            new_lst.append(new_txt)

        return new_lst

    def encode(self, text):
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        pattern = re.compile(GPT4_SPLIT_PATTERN)
        byte_txt_lst = [t.encode("utf-8") for t in re.findall(pattern, text)]
        for mer in self.merges:
            new_lst = []
            for text in byte_txt_lst:
                text = list(text)
                new_txt = []
                skip_next = False
                for pair in zip(text, text[1:]):
                    if skip_next:
                        skip_next = False
                        continue
                    if pair == mer:
                        new_txt.append(self.merges[pair])
                        skip_next = True
                    else:
                        new_txt.append(pair[0])
                if not skip_next:
                    new_txt.append(text[-1])
                new_lst.append(new_txt)
            byte_txt_lst = new_lst.copy()

        return [num for row in byte_txt_lst for num in row]

    def decode(self, ids):
        change = True
        while change:
            new_ids = []
            change = False
            for id in ids:
                if self.rev_merges.get(id):
                    new_ids += list(self.rev_merges[id])
                    change = True
                else:
                    new_ids.append(id)

            
            ids = new_ids.copy()
        text = bytes(ids).decode("utf-8")

        return text

    def visualize_merges(self):
        for pair, mer in self.merges.items():
            print(
                self.vocab[pair[0]],
                "+",
                self.vocab[pair[1]],
                "=",
                self.vocab[mer],
            )

In [89]:
regex_tokenizer = RegexTokenizer()
regex_tokenizer.train("hello uw ơ ????", 260)
en = regex_tokenizer.encode("hello uw ơ ????")
regex_tokenizer.decode(en)

'hello uw ơ ????'

In [84]:
text = "hello uw ơ ????"
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
pattern = re.compile(GPT4_SPLIT_PATTERN)
byte_txt_lst = [t.encode("utf-8") for t in re.findall(pattern, text)]
regex_tokenizer.merge_lst(byte_txt_lst, (63, 63)), byte_txt_lst

([[104, 101, 108, 108, 111], [32, 117, 119], [32, 198, 161], [32, 256, 256]],
 [b'hello', b' uw', b' \xc6\xa1', b' ????'])

In [85]:
regex_tokenizer.merges

{(63, 63): 256, (104, 101): 257, (257, 108): 258, (258, 108): 259}

In [86]:
regex_tokenizer.visualizer_merges()

b'?' + b'?' = b'??'
b'h' + b'e' = b'he'
b'he' + b'l' = b'hel'
b'hel' + b'l' = b'hell'


In [91]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) # get the same text back

In [93]:
ids

[15339,
 1917,
 12340,
 30,
 320,
 31495,
 230,
 75265,
 243,
 92245,
 16715,
 28509,
 4513,
 57037]